# Activity Notebook: Comparing Several Groups
### HBAT customer scenario

---

HBAT sells paper to magazine, newsprint and corporate customers, and management is worried about newer rivals. Do customers behave and feel differently depending on how long they have been with HBAT, which industry they serve or how they are supplied? If they do, which groups differ, by how much, and where should HBAT spend its attention?

This notebook compares the averages of three groups in one test (a one-way ANOVA), finds out which groups differ, and checks whether the test's assumptions hold. You already have the JMP results. Here you reproduce them in Python.

Every part follows the same five steps: **load and look, run the model, read the output in plain English, check against JMP, say it to a manager.**

---

### The AI rule

**Copilot is on.** You can ask it for code. Treat what it writes as a draft: check every result against the JMP result before you trust it. JMP is the ground truth.

---

### Before you start

- Open this notebook in your Codespace (`comparing-several-groups/` folder)
- The data file is `../data/Business Customer Data.xlsx`, already in your repository
- Run cells in order with **Shift+Enter**

## Setup: load, rename, set the group order

One block of imports, then the data. The Excel file has long column names with codes in front, such as `D1 - Customer Type`. We rename them once, here, so the rest of the notebook stays readable.

In [ ]:
import os
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.oneway import anova_oneway

print(os.listdir("../data"))   # check the data file is where we expect


def show_p(value):
    return "<0.0001" if value < 0.0001 else f"{value:.4f}"   # a tidy way to print small p-values

In [ ]:
df = pd.read_excel("../data/Business Customer Data.xlsx", sheet_name="B2B Data")   # the sheet with the 100 customers

short_names = {
    "Customer Name": "customer",
    "D1 - Customer Type": "customer_type",
    "D2 - Industry Type": "industry",
    "D3 - Firm Size": "firm_size",
    "D4 - Region": "region",
    "D5 - Distribution System": "channel",
    "P1 - Product Quality": "product_quality",
    "P2 - eCommerce Activities": "ecommerce",
    "P3 - Technical Support": "technical_support",
    "P4 - Complaint Resolution": "complaint_resolution",
    "P5 - Advertising": "advertising",
    "P6 - Product Line": "product_line",
    "P7 - Salesforce Image": "salesforce_image",
    "P8 - Competitive Pricing": "competitive_pricing",
    "P9 - Warranty & Claims": "warranty_claims",
    "P10 - New Products": "new_products",
    "P11 - Order & Billing": "order_billing",
    "P12 - Price Flexibility": "price_flexibility",
    "P13 - Delivery Speed": "delivery_speed",
    "OR1 - Satisfaction": "satisfaction",
    "OR2 - Likely to Recommend": "recommend",
    "OR3 - Likely to Purchase": "purchase_likelihood",
    "OR4 - Purchase Level": "purchase_level",
    "OR5 - Consider Strategic Alliance": "strategic_alliance",
}

df = df.rename(columns=short_names)
df.shape

Python sorts text labels alphabetically, which would put `1 to 5 years` first and `Less than 1 year` in the middle. We set the order ourselves so the groups run from newest to longest-served. This is the same job as setting a value order in JMP.

In [ ]:
df["customer_type"] = pd.Categorical(
    df["customer_type"],
    categories=["Less than 1 year", "1 to 5 years", "Over 5 years"],
    ordered=True,
)
df["industry"] = pd.Categorical(
    df["industry"],
    categories=["Corporate", "Magazine", "Newsprint"],
    ordered=True,
)
df["customer_type"].value_counts(sort=False)   # how many customers in each group, in order

---
## Demo: purchase level by customer type

Purchase level is scored from 1 to 100. Do newer customers buy less than long-standing ones?

### Load and look

In [ ]:
measure = "purchase_level"   # the number we are comparing
group = "customer_type"      # the groups we are comparing it across

levels = list(df[group].cat.categories)                         # the groups, in their set order
samples = [df.loc[df[group] == lv, measure] for lv in levels]   # one set of values per group

summary = df.groupby(group, observed=True)[measure].agg(["count", "mean", "std"]).round(3)
summary

### Run the model

In [ ]:
model = ols(f"{measure} ~ C({group})", data=df).fit()   # a one-way ANOVA, set up as a model
anova_table = sm.stats.anova_lm(model)                  # splits the spread into between-group and within-group
anova_table.round(3)

In [ ]:
F = anova_table.loc[f"C({group})", "F"]
p = anova_table.loc[f"C({group})", "PR(>F)"]
ss_between = anova_table.loc[f"C({group})", "sum_sq"]
ss_within = anova_table.loc["Residual", "sum_sq"]
eta_squared = ss_between / (ss_between + ss_within)   # the share of the spread the groups account for

print(f"F = {F:.2f}, p = {p:.2g}")
print(f"Eta squared = {eta_squared:.3f}: {eta_squared:.1%} of the spread in {measure} lines up with {group}")

The `Residual` row is the variation inside the groups. The `C(customer_type)` row is the variation between the groups. The F ratio is the first divided by the second, after each is averaged over its degrees of freedom.

In [ ]:
tukey = stats.tukey_hsd(*samples)       # all pairs, with the adjustment for making several comparisons
ci = tukey.confidence_interval()
rows = []
for i in range(len(levels)):
    for j in range(i + 1, len(levels)):
        rows.append({
            "pair": f"{levels[j]} minus {levels[i]}",
            "difference": round(-tukey.statistic[i, j], 2),   # second group in the pair minus the first
            "ci_low": round(-ci.high[i, j], 2),
            "ci_high": round(-ci.low[i, j], 2),
            "p (adjusted)": show_p(tukey.pvalue[i, j]),
        })
pd.DataFrame(rows)

### Read it in plain English

The three averages climb with the length of the relationship: about 49 for customers under a year, 59 for one to five years, and 67 for over five years. The F ratio is 105.80, which is far bigger than chance would produce, so the groups do not all share one average. Eta squared is 0.686: about 69% of the spread in purchase level lines up with how long the customer has been with HBAT.

The F ratio tells you that at least one group differs. It does not tell you which. The all-pairs table does that, and every pair differs here. Customers with over five years buy about 18 points more than the newest customers, with an interval of roughly 15 to 21 points.

These are different customers, not the same customers tracked over time. Long-standing customers are the ones who stayed, so the data cannot say that tenure causes higher purchase levels.

### Check against JMP

In [ ]:
jmp = {"F": 105.80, "eta_squared": 0.686}                  # the F Ratio and R-Square from the JMP demo
python = {"F": float(round(F, 2)), "eta_squared": float(round(eta_squared, 3))}
print("JMP:   ", jmp)
print("Python:", python)
print("Matches JMP" if python == jmp else "Does not match JMP: check the column names and the group order")

### Say it to a manager

*Newer customers buy noticeably less. Purchase level rises from about 49 for customers under a year to about 67 for those with over five years, and the length of the relationship lines up with about 69% of the spread in purchase level. These are different customers rather than the same ones tracked over time, so we can say the groups differ but not that tenure causes it.*

### Try this

Change `measure` to `"satisfaction"` in the first cell of this part and run the cells again from there. Does eta squared go up or down? Which pairs still differ?

### Commentary

1. In your own words, what does the F ratio compare?
2. A significant F ratio does not say which groups differ. Why not, and what do you run to find out?
3. Why does the all-pairs test adjust its p-values?

*Your commentary here*

---
## Demo: unequal spreads

The classic ANOVA assumes the groups have similar spreads. Product quality is scored from 1 to 10. Look at how the spread changes from group to group, then see what each test makes of it.

### Load and look

In [ ]:
measure = "product_quality"   # the number we are comparing
group = "customer_type"      # the groups we are comparing it across

levels = list(df[group].cat.categories)                         # the groups, in their set order
samples = [df.loc[df[group] == lv, measure] for lv in levels]   # one set of values per group

summary = df.groupby(group, observed=True)[measure].agg(["count", "mean", "std"]).round(3)
summary

### Run the model

In [ ]:
spread_ratio = summary["std"].max() / summary["std"].min()   # largest spread against smallest
print(f"Largest spread is {spread_ratio:.2f} times the smallest")

levene = stats.levene(*samples, center="mean")          # the version JMP calls Levene
brown_forsythe = stats.levene(*samples, center="median")   # the version JMP calls Brown-Forsythe
print(f"Levene p = {show_p(levene.pvalue)}")
print(f"Brown-Forsythe p = {show_p(brown_forsythe.pvalue)}")

In [ ]:
classic = anova_oneway(samples, use_var="equal")     # assumes similar spreads
welch = anova_oneway(samples, use_var="unequal")     # does not assume similar spreads

print(f"Classic ANOVA: F = {classic.statistic:.2f}, p = {classic.pvalue:.2g}")
print(f"Welch ANOVA:   F = {welch.statistic:.2f}, p = {welch.pvalue:.2g}")

In [ ]:
rows = []
for i in range(len(levels)):
    for j in range(i + 1, len(levels)):
        t = stats.ttest_ind(samples[j], samples[i], equal_var=False)   # Welch t-test: spreads may differ
        rows.append({
            "pair": f"{levels[j]} minus {levels[i]}",
            "difference": round(samples[j].mean() - samples[i].mean(), 2),
            "p (unadjusted)": show_p(t.pvalue),
            "p (Bonferroni)": show_p(min(1, t.pvalue * 3)),   # three pairs, so multiply by 3
        })
pd.DataFrame(rows)

### Read it in plain English

The spread in the longest-served group is about half the spread in the middle group, and both spread tests say that is not chance. So the equal-spread assumption does not hold for product quality. The Welch ANOVA does not need it. Its F ratio is 57.14, against 36.65 for the classic test. The numbers move, but the decision does not: the longest-served customers rate product quality about two points higher than either of the other groups, and the two newer groups do not differ from each other.

That is the usual outcome. When the spreads are similar, Welch and the classic test agree. When they are not, Welch is the one you can defend. It costs nothing to use it by default.

Tukey's all-pairs test also assumes similar spreads. The pairwise Welch t-tests above do not, and they show the same pattern. We multiplied each p-value by the number of pairs (a Bonferroni adjustment) so the table is fair about making several comparisons.

### When the shape is awkward

Satisfaction by customer type is a different kind of problem. The spread test is borderline, and one group has an awkward shape. Check each group's shape with a normality test, then compare the classic result with a rank-based test, which does not rely on the shape.

In [ ]:
measure = "satisfaction"
samples = [df.loc[df[group] == lv, measure] for lv in levels]   # the groups again, for satisfaction

for lv, values in zip(levels, samples):
    print(f"{lv}: Shapiro-Wilk p = {stats.shapiro(values).pvalue:.3f}")   # a low p means the shape looks odd

kruskal = stats.kruskal(*samples)    # Kruskal-Wallis: compares ranks, not averages
welch = anova_oneway(samples, use_var="unequal")
print(f"Kruskal-Wallis: H = {kruskal.statistic:.2f}, p = {kruskal.pvalue:.2g}")
print(f"Welch ANOVA:    F = {welch.statistic:.2f}, p = {welch.pvalue:.2g}")

The newest group has a low Shapiro-Wilk p-value, so its shape is a little odd. Look at the plots before you trust a normality test at around 33 customers per group. Here the rank-based test and the Welch ANOVA agree: satisfaction differs by customer type. When different methods give the same answer, you can report the result with more confidence.

### Check against JMP

In [ ]:
jmp = {"Welch F": 51.29, "Kruskal-Wallis H": 50.10}      # from the JMP demo
python = {"Welch F": float(round(welch.statistic, 2)), "Kruskal-Wallis H": float(round(kruskal.statistic, 2))}
print("JMP:   ", jmp)
print("Python:", python)
print("Matches JMP" if python == jmp else "Does not match JMP: check the group order and the measure")

### Say it to a manager

*Product quality is rated about two points higher by customers of over five years, and the two newer groups look alike. The groups have different spreads, so we used the Welch test, which gives the same decision with stronger evidence.*

### Commentary

1. Why did the classic F and the Welch F differ for product quality, and did any conclusion change?
2. When would you reach for the rank-based test instead of Welch?
3. Why do we look at plots before running a normality test?

*Your commentary here*

---
## Exercise: satisfaction by industry

Does satisfaction differ by the industry a customer serves? You have run the same steps on purchase level. This time, point the same code at a different measure and a different set of groups.

1. Change `measure` and `group` in the first cell below, then run all the cells in this part.
2. Compare your F ratio, eta squared and all-pairs table with what you found in JMP.
3. Decide which pairs differ, and whether the effect is big or small in business terms.

In [ ]:
measure = "purchase_level"   # change me
group = "customer_type"      # change me

levels = list(df[group].cat.categories)                         # the groups, in their set order
samples = [df.loc[df[group] == lv, measure] for lv in levels]   # one set of values per group

summary = df.groupby(group, observed=True)[measure].agg(["count", "mean", "std"]).round(3)
summary

In [ ]:
model = ols(f"{measure} ~ C({group})", data=df).fit()   # a one-way ANOVA, set up as a model
anova_table = sm.stats.anova_lm(model)                  # splits the spread into between-group and within-group
anova_table.round(3)

In [ ]:
F = anova_table.loc[f"C({group})", "F"]
p = anova_table.loc[f"C({group})", "PR(>F)"]
ss_between = anova_table.loc[f"C({group})", "sum_sq"]
ss_within = anova_table.loc["Residual", "sum_sq"]
eta_squared = ss_between / (ss_between + ss_within)   # the share of the spread the groups account for

print(f"F = {F:.2f}, p = {p:.2g}")
print(f"Eta squared = {eta_squared:.3f}: {eta_squared:.1%} of the spread in {measure} lines up with {group}")

In [ ]:
tukey = stats.tukey_hsd(*samples)       # all pairs, with the adjustment for making several comparisons
ci = tukey.confidence_interval()
rows = []
for i in range(len(levels)):
    for j in range(i + 1, len(levels)):
        rows.append({
            "pair": f"{levels[j]} minus {levels[i]}",
            "difference": round(-tukey.statistic[i, j], 2),   # second group in the pair minus the first
            "ci_low": round(-ci.high[i, j], 2),
            "ci_high": round(-ci.low[i, j], 2),
            "p (adjusted)": show_p(tukey.pvalue[i, j]),
        })
pd.DataFrame(rows)

In [ ]:
jmp_f = None      # type the F Ratio from your JMP result
jmp_eta = None    # type the R-Square from your JMP result
if jmp_f is None:
    print("Type your JMP numbers above, then run this cell again.")
else:
    matches = round(F, 2) == jmp_f and round(eta_squared, 3) == jmp_eta
    print("Matches JMP" if matches else "Does not match JMP: check measure and group")

### Commentary

1. Which pairs of industries differ, and which do not? Quote the difference and its interval for one pair.
2. The effect is real. Is it large? Use eta squared and the size of the differences in points of satisfaction.
3. What would you want to know before telling HBAT to treat corporate customers differently?

*Your commentary here*

---
## Exercise: technical support by customer type, with Copilot

Technical support is scored from 1 to 10. Do customers of different tenure rate it differently?

This time you write the prompt. Ask Copilot for a one-way ANOVA with an all-pairs comparison and the eta squared, and say which data frame and columns to use. A good prompt names the data frame (`df`), the two columns, the libraries (`scipy` and `statsmodels`), and what to print. Ask it to store the F ratio in a variable called `F`.

1. Write your prompt in the cell below.
2. Paste the code Copilot gives you into the code cell and run it.
3. Check its F ratio and p-value against your JMP result. If they do not match, work out why before you trust anything else it printed.

*Your prompt here*

In [ ]:
# Paste Copilot's code here

### Commentary

1. What did Copilot get right, and what did you have to fix or check?
2. Does tenure change how customers rate technical support? Say so in words, including what a non-significant result does and does not mean.
3. After a run of strong results, what does this one tell HBAT about where to spend its attention?

*Your commentary here*

---
## Find the bug

Copilot wrote this code to label the all-pairs results for purchase level by customer type. It runs without any error. Compare each number with the all-pairs results from the first demo, then find the problem and fix it.

In [ ]:
# Code from Copilot: all-pairs comparison of purchase level by customer type
raw = pd.read_excel("../data/Business Customer Data.xlsx", sheet_name="B2B Data").rename(columns=short_names)

labels = ["Less than 1 year", "1 to 5 years", "Over 5 years"]
samples = [g["purchase_level"] for _, g in raw.groupby("customer_type")]
result = stats.tukey_hsd(*samples)

for i in range(3):
    for j in range(i + 1, 3):
        print(f"{labels[j]} minus {labels[i]}: {-result.statistic[i, j]:.2f}   p = {show_p(result.pvalue[i, j])}")

### Commentary

1. What was wrong with the labels, and how did you spot it?
2. Why does this kind of mistake survive a run that shows no error?
3. What would you add to a prompt so Copilot is less likely to make it?

*Your commentary here*

---
## Say it to a manager

Write two sentences for a manager who has not seen any statistics. Use the result you found in the exercises. Say what differs, or does not, by how much, and one thing you would not claim.

*Your two sentences here*

---
## Stretch

A blank space for you and Copilot. Some ideas:

- Draw a box plot of purchase level by customer type. If Python says `matplotlib` is missing, run `%pip install matplotlib` in a cell and try again.
- Run the full set of steps for another rating, such as `order_billing` or `price_flexibility`, by customer type.
- Build one table of the average of every rating by customer type, and look for the ones that move.
- Ask Copilot how to compare the groups with a trimmed or transformed measure, and check the answer against what you know.

In [ ]:
# Stretch